# 07. Feature Importance

Với mỗi model và mỗi feature, tính 3 chỉ số trên 800 dòng train (seed 1042):

| Phương pháp | Cách tính |
|---|---|
| LIME | `LimeTabularExplainer(mode="regression")` trên 30 dòng, trung bình trị tuyệt đối weight, lấy trung bình cash và card |
| SHAP | `KernelExplainer`, background 40 dòng, giải thích 80 dòng, `nsamples=129`, trung bình trị tuyệt đối, lấy trung bình cash và card |
| Permutation | Mức giảm `qini_cash + qini_card` khi hoán vị feature, trung bình 3 lần |

$$FI^{(k)}_j = \frac{1}{5}\sum_{m=1}^{5} FI^{(k)}_{j,m}, \qquad FI^{final}_j = \frac{FI^{(lime)}_j + FI^{(shap)}_j + FI^{(permutation)}_j}{3}$$

In [ ]:
import sys
sys.path.append("..")

import joblib
from src import config as C
from src.explain import feature_importance
from src.plots import importance_bar
from src.preprocessing import load_model_data

X, W, y_cash, y_card, feat_cols = load_model_data()
bundle = joblib.load(C.RESULTS_DIR / "inference" / "ensemble_models.joblib")

In [ ]:
summary, per_model = feature_importance(bundle["models"], X, W, y_cash, y_card, feat_cols)
summary.head(20)

In [ ]:
out = C.RESULTS_DIR / "feature_importance"
out.mkdir(exist_ok=True)
summary.to_csv(out / "feature_importance.csv", index=False)
for name, detail in per_model.items():
    detail.to_csv(out / f"{name}_feature_importance.csv", index=False)

In [ ]:
importance_bar(summary, top=15, save=C.FIGURES_DIR / "feature_importance_top15.png")